# Proyecto final: Asistente Vissotia con Gemini, RAG y LangGraph

Este notebook presenta un asistente documental público sobre Vissotia. Utiliza **Gemini** para generación y embeddings, **ChromaDB** como base vectorial y **LangGraph** para el flujo conversacional con memoria.

El corpus contiene exclusivamente información pública, aprobada o simulada. El asistente no accede a cuentas, clientes, proyectos ni inmuebles reales.

## 1. Preparación reproducible

Antes de ejecutar el notebook:

1. Ejecutar `uv sync --dev` en la raíz del proyecto.
2. Copiar `.env.example` como `.env`.
3. Añadir `GEMINI_API_KEY` únicamente al `.env` local.

La clave nunca se imprime ni se almacena en las salidas del notebook.

In [1]:
from __future__ import annotations

import sys
import uuid
from collections import Counter
from decimal import Decimal
from pathlib import Path

from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / 'src').is_dir():
        PROJECT_ROOT = candidate
    else:
        raise RuntimeError('Ejecuta el notebook desde la raíz del proyecto o desde la carpeta notebooks/.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Proyecto: {PROJECT_ROOT.name}')

Proyecto: Proyecto_IA_generativa


In [2]:
from vissotia_assistant.calculators import (
    RenovateAndSellInput,
    RentalSimulationInput,
    simulate_renovate_and_sell,
    simulate_rental_investment,
)
from vissotia_assistant.config import Settings
from vissotia_assistant.corpus import (
    build_chunks,
    discover_approved_sources,
    write_chunks_jsonl,
)
from vissotia_assistant.graph import build_graph
from vissotia_assistant.rag import BASE_SYSTEM_PROMPT, RagService
from vissotia_assistant.vectorstore import (
    create_rate_limited_gemini_embeddings,
    create_vector_store,
    open_vector_store,
)

settings = Settings.from_env()
settings.require_api_key()
print('Configuración cargada. La clave de Gemini está disponible y no se mostrará.')
print(f'Modelo de chat: {settings.chat_model}')
print(f'Modelo de embeddings: {settings.embedding_model}')
print(f'Dimensiones: {settings.embedding_dimensions}')

Configuración cargada. La clave de Gemini está disponible y no se mostrará.
Modelo de chat: gemini-3.5-flash-lite
Modelo de embeddings: gemini-embedding-2
Dimensiones: 768


## 2. Corpus público, validación y chunking

Los documentos Markdown incorporan metadatos de procedencia, audiencia, alcance, estado y URL. Solo se indexan documentos con `status: approved`. La validación rechaza contenido vacío, rutas locales, tipos de fuente no permitidos y posibles secretos.

La segmentación se realiza primero por encabezados Markdown y después con un divisor recursivo de **1.000 caracteres** y **150 caracteres de solapamiento**. Esta combinación mantiene secciones comprensibles sin crear bloques demasiado extensos.

In [3]:
source_root = PROJECT_ROOT / 'data' / 'public_sources'
source_paths = discover_approved_sources(source_root)
chunks = build_chunks(source_root, chunk_size=1_000, chunk_overlap=150)
counts = Counter(str(chunk.metadata['document_id']) for chunk in chunks)

print(f'Documentos aprobados: {len(source_paths)}')
print(f'Chunks generados: {len(chunks)}')
for document_id, count in sorted(counts.items()):
    print(f'- {document_id}: {count} chunks')

Documentos aprobados: 9
Chunks generados: 123
- vissotia_01_que_es: 7 chunks
- vissotia_02_funcionalidades: 8 chunks
- vissotia_03_onboarding_profesionales: 10 chunks
- vissotia_04_onboarding_particulares_inversores: 15 chunks
- vissotia_05_fuentes_contexto: 15 chunks
- vissotia_06_marketing_arquetipo: 8 chunks
- vissotia_07_seguridad_limites: 9 chunks
- vissotia_08_casos_simulados: 24 chunks
- vissotia_09_glosario_faq: 27 chunks


In [4]:
sample = chunks[0]
display(Markdown('### Ejemplo de chunk'))
print(sample.page_content[:700])
display(Markdown('### Metadatos públicos conservados'))
for key in ('document_id', 'title', 'source_type', 'header_1', 'header_2', 'status'):
    value = sample.metadata.get(key) or ''
    print(f'{key}: {value}')

### Ejemplo de chunk

Guía conceptual de inicio para profesionales > Objetivo de esta guía

Esta guía explica cómo puede abordarse un proyecto inmobiliario con Vissotia desde una perspectiva conceptual. No reproduce la interfaz ni describe botones, menús, arquitectura o procesos internos.  
El recorrido no sustituye el criterio del profesional. Su finalidad es mostrar cómo se conectan información, contexto, estrategia, contenidos y seguimiento.


### Metadatos públicos conservados

document_id: vissotia_03_onboarding_profesionales
title: Guía conceptual de inicio para profesionales
source_type: approved_public
header_1: Guía conceptual de inicio para profesionales
header_2: Objetivo de esta guía
status: approved


## 3. ChromaDB y Gemini Embeddings

El índice es persistente y no se publica en GitHub. Si no existe localmente, esta sección guarda el corpus procesado, obtiene embeddings de Gemini en lotes y crea la colección Chroma. Si ya existe, la reutiliza.

In [5]:
processed_path = PROJECT_ROOT / 'data' / 'processed' / 'chunks.jsonl'
write_chunks_jsonl(chunks, processed_path)

index_exists = settings.chroma_directory.exists() and any(settings.chroma_directory.iterdir())
if index_exists:
    vector_store = open_vector_store(settings=settings)
    if vector_store._collection.count() == 0:
        raise RuntimeError('La colección existe pero está vacía. Reconstruye data/chroma.')
    print('Se reutiliza el índice Chroma existente.')
else:
    embeddings = create_rate_limited_gemini_embeddings(settings)
    vector_store = create_vector_store(
        chunks,
        embedding=embeddings,
        collection_name=settings.chroma_collection,
        persist_directory=settings.chroma_directory,
    )
    print('Índice Chroma creado con Gemini Embeddings.')

print(f'Colección: {settings.chroma_collection}')
print(f'Vectores almacenados: {vector_store._collection.count()}')

Se reutiliza el índice Chroma existente.
Colección: vissotia_public_mvp
Vectores almacenados: 123


## 4. Evaluación del retriever antes de conectar el LLM

Se comprueba que las consultas recuperan documentos esperados antes de generar respuestas. Se utiliza `k=5`; posteriormente el servicio RAG aplica un umbral de relevancia y un corte relativo para evitar contexto débil.

In [6]:
retrieval_queries = [
    '¿Qué es Vissotia y qué problema resuelve?',
    '¿Catastro demuestra quién es el propietario de una vivienda?',
    '¿Cómo se crea un arquetipo de comprador sin perfilar personas reales?',
    'Quiero simular una compra para alquilar',
]

for query in retrieval_queries:
    print(f'\nCONSULTA: {query}')
    results = vector_store.similarity_search_with_relevance_scores(query, k=3)
    for rank, (document, score) in enumerate(results, start=1):
        section = document.metadata.get('header_2') or document.metadata.get('header_1', '')
        title = document.metadata.get('title', 'Fuente')
        print(f'{rank}. {score:.3f} | {title} | {section}')


CONSULTA: ¿Qué es Vissotia y qué problema resuelve?


1. 0.697 | Qué es Vissotia | Qué problema busca resolver
2. 0.666 | Qué es Vissotia | Descripción general
3. 0.643 | Glosario y preguntas frecuentes | Preguntas frecuentes

CONSULTA: ¿Catastro demuestra quién es el propietario de una vivienda?


1. 0.645 | Fuentes y contexto de datos | Dirección General del Catastro
2. 0.614 | Fuentes y contexto de datos | Dirección General del Catastro
3. 0.547 | Seguridad, privacidad y límites del Asistente Vissotia | Información sobre inmuebles

CONSULTA: ¿Cómo se crea un arquetipo de comprador sin perfilar personas reales?


1. 0.679 | Marketing inmobiliario y arquetipo de comprador objetivo | Diferencia entre arquetipo y perfilado real
2. 0.624 | Marketing inmobiliario y arquetipo de comprador objetivo | Proceso conceptual
3. 0.621 | Marketing inmobiliario y arquetipo de comprador objetivo | Información que puede utilizar un arquetipo simulado

CONSULTA: Quiero simular una compra para alquilar


1. 0.604 | Guía conceptual de inicio para particulares e inversores | Recorrido de un inversor
2. 0.593 | Casos de uso simulados | Caso 4. Compra ficticia para alquiler
3. 0.558 | Casos de uso simulados | Caso 4. Compra ficticia para alquiler


## 5. System prompt y decisiones de diseño

El prompt define al asistente como una demostración pública y obliga a fundamentar las afirmaciones sobre Vissotia exclusivamente en el contexto recuperado. Sus decisiones principales son:

- impedir la invención de funciones, precios, integraciones o resultados;
- declarar insuficiencia cuando el corpus no responde;
- no acceder ni solicitar datos personales o de operaciones reales;
- identificar los ejemplos como simulados;
- separar el texto de respuesta de las fuentes mostradas por la interfaz;
- adaptar la explicación al perfil sin cambiar los hechos;
- utilizar el historial para resolver continuaciones breves.

La temperatura se configura en **0** para reducir variabilidad en respuestas documentales. La generación queda además acotada por reglas explícitas y el contexto RAG.

In [7]:
print(BASE_SYSTEM_PROMPT)

Eres el Asistente Vissotia, un asistente público y demostrativo.

Responde en español y utiliza únicamente el CONTEXTO PÚBLICO recuperado para realizar
afirmaciones sobre Vissotia, sus funciones, fuentes y límites.

Reglas obligatorias:
- No inventes funcionalidades, precios, planes, integraciones, plazos ni resultados.
- Si el contexto no responde suficientemente, dilo con claridad.
- No afirmes que puedes acceder a cuentas, clientes, proyectos o inmuebles reales.
- No solicites datos personales ni documentación de una operación real.
- Los ejemplos inmobiliarios deben identificarse como ejemplos simulados.
- No proporciones asesoramiento jurídico, fiscal, técnico, financiero o de inversión personalizado.
- No incluyas una bibliografía dentro del texto: la interfaz mostrará las fuentes por separado.
- No menciones IDs, puntuaciones, rutas locales, embeddings ni detalles internos del RAG.
- Responde primero a la pregunta y, solo si aporta valor, termina con una sugerencia breve.
- El p

## 6. Agente LangGraph con RAG y memoria

El grafo valida el perfil, clasifica la petición y deriva hacia RAG, una herramienta determinista, una respuesta segura o una aclaración. `InMemorySaver` conserva los mensajes asociados a cada `thread_id`, por lo que dos conversaciones permanecen separadas.

In [8]:
rag_service = RagService(settings=settings)
graph = build_graph(rag_service=rag_service)
display(Markdown(f'```mermaid\n{graph.get_graph().draw_mermaid()}\n```'))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validate_input(validate_input)
	route_request(route_request)
	rag(rag)
	rental_form(rental_form)
	renovate_sell_form(renovate_sell_form)
	safe_response(safe_response)
	clarify(clarify)
	general(general)
	__end__([<p>__end__</p>]):::last
	__start__ --> validate_input;
	route_request -.-> clarify;
	route_request -.-> general;
	route_request -.-> rag;
	route_request -.-> renovate_sell_form;
	route_request -.-> rental_form;
	route_request -.-> safe_response;
	validate_input --> route_request;
	clarify --> __end__;
	general --> __end__;
	rag --> __end__;
	renovate_sell_form --> __end__;
	rental_form --> __end__;
	safe_response --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## 7. Función de interacción textual

La misma función permite conversar desde el notebook. Para mantener memoria se reutiliza el `thread_id`; para iniciar una conversación independiente se utiliza uno nuevo.

In [9]:
def chat(
    question: str,
    *,
    profile: str = 'owner',
    thread_id: str | None = None,
) -> dict:
    thread_id = thread_id or str(uuid.uuid4())
    result = graph.invoke(
        {
            'messages': [HumanMessage(content=question)],
            'query': question,
            'user_profile': profile,
        },
        {'configurable': {'thread_id': thread_id}},
    )
    display(Markdown(f'**Pregunta:** {question}'))
    display(Markdown(result.get('answer', '')))
    sources = result.get('sources', [])
    if sources:
        display(Markdown('**Fuentes recuperadas:**'))
        for source in sources:
            section = f" — {source.get('section', '')}" if source.get('section') else ''
            print(f"- {source.get('title', 'Fuente')}{section}")
    return result

## 8. Cinco preguntas documentadas

Las siguientes pruebas cubren definición del producto, beneficio profesional, límites de una fuente, marketing responsable y privacidad. Cada ejemplo utiliza una conversación independiente para evitar contaminación de memoria.

In [10]:
examples = [
    ('¿Qué es Vissotia?', 'owner'),
    ('¿Qué aporta Vissotia a una agencia inmobiliaria?', 'professional'),
    ('¿Catastro demuestra quién es el propietario de una vivienda?', 'owner'),
    ('¿Cómo genera Vissotia un arquetipo de comprador para redactar anuncios?', 'professional'),
    ('¿Puedes decirme qué inmuebles tengo guardados en Vissotia?', 'professional'),
]

example_results = []
for question, profile in examples:
    example_results.append(chat(question, profile=profile))
    display(Markdown('---'))

<PROJECT_ROOT>/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Pregunta:** ¿Qué es Vissotia?

Vissotia es una plataforma digital que ayuda a organizar y llevar a cabo operaciones inmobiliarias. Funciona como un punto de encuentro central (lo que llamamos una *Single Source of Truth* o fuente única de verdad) para reunir toda la información, las conversaciones y las tareas que normalmente están repartidas en muchos sitios diferentes, como papeles sueltos o distintas aplicaciones.

Para lograrlo, la plataforma une cinco elementos clave:
1. **Datos:** la información sobre el inmueble y la zona en la que se encuentra.
2. **Contexto:** los detalles necesarios para entender bien esos datos.
3. **Inteligencia:** la ayuda de la inteligencia artificial para ordenar toda la información.
4. **Supervisión humana:** las decisiones importantes siempre dependen de las personas.
5. **Acción y seguimiento:** el control de los pasos que se van dando en la operación.

Su objetivo es evitar que se pierdan datos o que haya confusiones al gestionar todo por separado, logrando que tareas como calcular un valor, preparar la estrategia, hacer los materiales de marketing o revisar documentos estén conectadas entre sí de forma ordenada.

¿Te gustaría que te explique cuáles son los siguientes pasos para conocer mejor cómo funciona la plataforma?

**Fuentes recuperadas:**

- Glosario y preguntas frecuentes — Preguntas frecuentes
- Qué es Vissotia — Descripción general
- Qué es Vissotia — Qué problema busca resolver


---

... (17 outputs left out)

---

**Pregunta:** ¿Puedes decirme qué inmuebles tengo guardados en Vissotia?

Este chat público no está conectado a cuentas, clientes, proyectos ni inmuebles registrados en Vissotia. Puedo explicarte el funcionamiento mediante un ejemplo simulado.

---

## 9. Prueba explícita de memoria

La segunda pregunta depende de la primera: la expresión «la oficial» debe interpretarse como una tasación oficial. Ambos turnos utilizan el mismo identificador.

In [11]:
memory_thread = f'memory-{uuid.uuid4()}'
_ = chat('Explícame qué es una valoración orientativa.', profile='owner', thread_id=memory_thread)
_ = chat('¿Y sustituye a la oficial?', profile='owner', thread_id=memory_thread)

<PROJECT_ROOT>/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Pregunta:** Explícame qué es una valoración orientativa.

Una valoración orientativa es una referencia informativa que te ayuda a analizar una vivienda y a preparar una conversación, por ejemplo, al explorar el mercado inmobiliario. 

Para entenderla mejor, debes tener en cuenta lo siguiente:
- **No es una tasación oficial:** Por lo tanto, no tiene validez legal ni oficial.
- **No garantiza resultados:** No asegura ni el precio exacto al que se venderá una vivienda ni la rentabilidad que podría obtener un inversor.
- **Requiere revisión:** Es necesario comprobar los datos utilizados, ver si hay información sin confirmar y tener en cuenta si existen características singulares que deban revisarse con criterio profesional.

¿Te gustaría saber qué preguntas conviene hacer antes de utilizar una valoración orientativa?

**Fuentes recuperadas:**

- Guía conceptual de inicio para particulares e inversores — 4. Comprender una valoración orientativa
- Glosario y preguntas frecuentes — Glosario


<PROJECT_ROOT>/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Pregunta:** ¿Y sustituye a la oficial?

No, una valoración orientativa no sustituye a una tasación oficial ni garantiza el precio de una operación.

¿Quieres que te explique cómo se utiliza normalmente este tipo de estimación orientativa?

**Fuentes recuperadas:**

- Glosario y preguntas frecuentes — Preguntas frecuentes


## 10. Información insuficiente y consulta fuera del dominio

El agente no debe completar con conocimiento general las afirmaciones sobre Vissotia ni inventar condiciones comerciales. También debe reconocer cuando una consulta no pertenece al dominio documental.

In [12]:
_ = chat('¿Qué precio tiene el plan empresarial y cuántos usuarios admite?', profile='professional')
display(Markdown('---'))
_ = chat('¿Cómo se prepara una paella?', profile='owner')

<PROJECT_ROOT>/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Pregunta:** ¿Qué precio tiene el plan empresarial y cuántos usuarios admite?

El contexto público disponible no incluye información sobre precios, planes comerciales ni el número de usuarios que admite Vissotia. 

Si lo deseas, puedes consultar los enlaces externos disponibles para contactar con soporte o solicitar más información sobre planes.

**Fuentes recuperadas:**

- Glosario y preguntas frecuentes — Preguntas frecuentes
- Seguridad, privacidad y límites del Asistente Vissotia — Contacto y siguientes pasos
- Qué es Vissotia — Qué problema busca resolver
- Funcionalidades y beneficios de Vissotia — Valoración asistida por IA


---

<PROJECT_ROOT>/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Pregunta:** ¿Cómo se prepara una paella?

El contexto disponible no contiene información sobre cómo se prepara una paella. Vissotia es un asistente público centrado en guiar y ordenar información para particulares e inversores en el ámbito inmobiliario, por lo que no dispongo de recetas de cocina.

Si necesitas ayuda para empezar a ordenar información antes de analizar una vivienda o preparar una posible venta o inversión, dime qué objetivo tienes y te explico los primeros pasos.

**Fuentes recuperadas:**

- Guía conceptual de inicio para particulares e inversores — 1. Empezar por el objetivo


## 11. Herramientas deterministas

Gemini no realiza estas operaciones. Las herramientas reciben supuestos explícitos, validan los campos y devuelven resultados reproducibles junto con una advertencia.

In [13]:
rental_result = simulate_rental_investment(
    RentalSimulationInput(
        purchase_price=Decimal('200000'),
        acquisition_costs=Decimal('20000'),
        renovation_costs=Decimal('10000'),
        other_initial_costs=Decimal('0'),
        monthly_rent=Decimal('1000'),
        occupied_months_per_year=Decimal('11'),
        annual_operating_costs=Decimal('2000'),
        annual_financing_costs=Decimal('3000'),
    )
)
display(Markdown('### Simulación orientativa: compra para alquiler'))
rental_result.model_dump()

### Simulación orientativa: compra para alquiler

{'label': 'Simulación orientativa',
 'total_initial_investment': Decimal('230000.00'),
 'annual_gross_income': Decimal('11000.00'),
 'annual_net_income_before_financing': Decimal('9000.00'),
 'annual_cash_flow_after_financing': Decimal('6000.00'),
 'gross_yield_pct': Decimal('4.78'),
 'net_yield_before_financing_pct': Decimal('3.91'),
 'disclaimer': 'Este ejemplo no constituye una valoración, previsión ni recomendación de inversión. Verifica los datos y consulta a profesionales cualificados antes de tomar una decisión real.'}

In [14]:
flip_result = simulate_renovate_and_sell(
    RenovateAndSellInput(
        purchase_price=Decimal('150000'),
        acquisition_costs=Decimal('15000'),
        renovation_budget=Decimal('30000'),
        renovation_contingency=Decimal('5000'),
        holding_costs=Decimal('4000'),
        financing_costs=Decimal('6000'),
        selling_costs=Decimal('10000'),
        other_costs=Decimal('0'),
        estimated_sale_price=Decimal('250000'),
    )
)
display(Markdown('### Simulación orientativa: compra, reforma y venta'))
flip_result.model_dump()

### Simulación orientativa: compra, reforma y venta

{'label': 'Simulación orientativa',
 'total_project_cost': Decimal('220000.00'),
 'estimated_sale_price': Decimal('250000.00'),
 'orientative_result': Decimal('30000.00'),
 'margin_on_sale_pct': Decimal('12.00'),
 'return_on_total_cost_pct': Decimal('13.64'),
 'disclaimer': 'Este ejemplo no constituye una valoración, previsión ni recomendación de inversión. Verifica los datos y consulta a profesionales cualificados antes de tomar una decisión real.'}

## 12. Celda libre de conversación

Modifica la pregunta, el perfil o el identificador para continuar probando el agente.

In [15]:
interactive_thread = f'interactive-{uuid.uuid4()}'
question = '¿Qué ventajas ofrece Vissotia a un propietario?'
_ = chat(question, profile='owner', thread_id=interactive_thread)

<PROJECT_ROOT>/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Pregunta:** ¿Qué ventajas ofrece Vissotia a un propietario?

Vissotia ofrece a un propietario (en el marco de un ejemplo simulado) la ventaja de reunir en un mismo lugar toda la información, los documentos, la estrategia y las tareas de un proyecto inmobiliario, evitando que se gestionen de forma separada o fragmentada. 

Esto se traduce en varias ventajas prácticas:
1. **Conservar el contexto:** Significa que cuando cambian los datos de un inmueble (como una característica o un precio planteado), las explicaciones y los materiales que se preparen después mantienen una base coherente y ordenada.
2. **Reducir duplicidades y errores:** Al centralizarlo todo en una capa común, se evitan confusiones y se facilita una comunicación más clara.
3. **Apoyo en marketing:** Ayuda a redactar primeros borradores de mensajes comerciales basados en las características del inmueble y en un tipo de comprador ideal (una hipótesis comercial general, sin usar datos de personas reales), adaptando el contenido a diferentes canales para que el equipo o el
 propietario puedan revisarlo antes de publicarlo.

*¿Te gustaría conocer cómo se relacionan los datos del inmueble con la estrategia comercial en la plataforma?*

**Fuentes recuperadas:**

- Funcionalidades y beneficios de Vissotia — Una visión conectada del proyecto inmobiliario
- Qué es Vissotia — Qué problema busca resolver
- Qué es Vissotia — Descripción general
- Funcionalidades y beneficios de Vissotia — Marketing inmobiliario asistido por IA


## 13. Conclusiones y límites

El proyecto demuestra un pipeline completo de IA generativa con recuperación: corpus aprobado, chunking auditable, Gemini Embeddings, ChromaDB, RAG, Gemini como LLM, LangGraph y memoria temporal. La interfaz Streamlit reutiliza los mismos componentes.

Limitaciones principales:

- el MVP no consulta APIs externas en tiempo real;
- no accede a datos privados ni guarda conversaciones entre sesiones;
- las simulaciones son orientativas y no constituyen asesoramiento;
- la calidad depende de la cobertura y actualización del corpus público;
- cualquier contenido generado requiere revisión humana.